In [ ]:
import geopandas as gpd
import pandas as pd
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input'
output_step1_path='../../Data/output/step-1'
output_step2_path='../../Data/output/step-2'
output_step3_path='../../Data/output/step-3'

#Read the files
index_walkability = gpd.read_parquet(f'{output_step3_path}/step3_index.parquet')
index_walkability = index_walkability.to_crs(operation_crs)

zones_girec = gpd.read_file(f'{input_file_path}/network_agreg/GEO_GIREC-SHP/GEO_GIREC.shp')
zones_girec = zones_girec.to_crs(operation_crs)

agglo_carreau = gpd.read_file(f'{input_file_path}/network_agreg/AGGLO_CARREAU_200-SHP/AGGLO_CARREAU_200.shp')
agglo_carreau = agglo_carreau.to_crs(operation_crs)

zones_communes = gpd.read_file(f'{input_file_path}/network_agreg/CAD_COMMUNE-SHP/CAD_COMMUNE.shp')
zones_communes = zones_communes.to_crs(operation_crs)

zones_communes_GE_fusionnee = gpd.read_file(f'{input_file_path}/network_agreg/CAD_COMMUNE-SHP/CAD_COMMUNES_GE_fusionnee.shp')
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.to_crs(operation_crs)


**GIREC**

In [ ]:
# Spatial join
segments_girec = gpd.sjoin(index_walkability, zones_girec, how="inner", predicate="within")

# Columns to aggregate
cols = index_walkability.columns.to_list()

def weighted_mean(df, cols, weight_col):
    return (df[cols].multiply(df[weight_col], axis=0).sum() / df[weight_col].sum())

cols_to_agg = cols[3:]  # tes colonnes d'indicateurs

# Calcul pondéré
girec_stats = (
    segments_girec
        .groupby("OBJECTID")
        .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
        .reset_index()
)

# Merge back with zones_mmt polygons
zones_girec = zones_girec.merge(girec_stats, on="OBJECTID", how="left")

#Drop nan values 
zones_girec = zones_girec.dropna(subset=["indice_marchabilite"])

In [ ]:
zones_girec.head()

**Carreau 200**

In [ ]:
# Spatial join: assign each segment to a carreau (grid cell)
segments_carreau = gpd.sjoin(index_walkability, agglo_carreau, how="inner", predicate="within")

# Aggregate by mean
carreau_stats = (
    segments_carreau
    .groupby("GRID_ID")
    .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
    .reset_index()
)

# Merge back to grid polygons
agglo_carreau = agglo_carreau.merge(carreau_stats, on="GRID_ID", how="left")

# Drop rows with missing values (optional)
agglo_carreau = agglo_carreau.dropna(subset=["indice_marchabilite"])

In [ ]:
segments_carreau.head()

**[Communes](https://sitg.ge.ch/donnees/cad-commune)**

In [ ]:
zones_communes.head()

In [ ]:
# Spatial join: assign each segment to a carreau (grid cell)
segments_communes = gpd.sjoin(index_walkability, zones_communes, how="inner", predicate="within")

# Aggregate by mean
communes_stats = (
    segments_communes
    .groupby("OBJECTID")
    .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
    .reset_index()
)

# Merge back to grid polygons
zones_communes = zones_communes.merge(communes_stats, on="OBJECTID", how="left")

# Drop rows with missing values (optional)
zones_communes = zones_communes.dropna(subset=["indice_marchabilite"])

In [ ]:
zones_communes.head()

*Communes avec Genève fusionnée (1 seul polygone au lieu de 4 différents)*

In [ ]:
# Spatial join: assign each segment to a carreau (grid cell)
segments_communes_GE_fusionnee = gpd.sjoin(index_walkability, zones_communes_GE_fusionnee, how="inner", predicate="within")

# Aggregate by mean
communes_GE_fusionnee_stats = (
    segments_communes_GE_fusionnee
    .groupby("OBJECTID")
    .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
    .reset_index()
)

# Merge back to grid polygons
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.merge(communes_GE_fusionnee_stats, on="OBJECTID", how="left")

# Drop rows with missing values (optional)
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.dropna(subset=["indice_marchabilite"])

In [ ]:
#save the file 
zones_girec.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_girec.gpkg"), driver="GPKG")
zones_girec.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_girec.parquet')

agglo_carreau.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_carreau200.gpkg"), driver="GPKG")
agglo_carreau.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_carreau200.parquet')

zones_communes.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_communes.gpkg"), driver="GPKG")
zones_communes.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_communes.parquet')

zones_communes_GE_fusionnee.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_communes_GE_fusionnee.gpkg"), driver="GPKG")
zones_communes_GE_fusionnee.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_communes_GE_fusionnee.parquet')

**[FREQUENCE CRIMINALITE GENEVE 2024](https://statistique.ge.ch/atlas/index.php#c=indicator&view=map3)**

In [ ]:
criminalite_communes = pd.read_csv(f'{input_file_path}/STAT/CRIMINALITE/CRIMINALITE_GE_2024.csv', sep=";", header=2)

criminalite_communes = criminalite_communes.rename(columns={"Loi sur les stupéfiants (LStup) : fréquence d'infractions 2024":"freq_LStup_infra","Code pénal (CP) : fréquence d'infractions 2024":"freq_CP_infra", "Loi sur les étrangers et l’intégration (LEI) : fréquence d'infractions 2024":"freq_LEI_infra"})

#convert string to float
colonnes = ['freq_LStup_infra', 'freq_CP_infra', 'freq_LEI_infra']

criminalite_communes[colonnes] = criminalite_communes[colonnes].apply(pd.to_numeric, errors='coerce')

criminalite_communes['freq_crim_mean'] = criminalite_communes[colonnes].mean(axis=1)

#print(criminalite_communes[colonnes].dtypes)

In [ ]:
criminalite_communes.head()

In [ ]:
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.merge(
    criminalite_communes,
    left_on="NO_COM_FED",
    right_on="Code",
    how="left" 
)

In [ ]:
zones_communes_GE_fusionnee.head()

In [ ]:
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.drop(columns=['Code', 'Libellé'])

In [ ]:
zones_communes_GE_fusionnee.head()

**[TAUX MOTORISATION CANTON GENEVE](https://statistique.ge.ch/atlas/index.php#c=indicator&view=map3)**

In [ ]:
taux_motorisation_communes = pd.read_csv(f'{input_file_path}/STAT/TAUX_MOTORISATION/taux_motorisation_2024.csv', sep=";", header=2)

In [ ]:
taux_motorisation_communes.head()

In [ ]:
taux_motorisation_communes = taux_motorisation_communes.rename(columns={"Taux de motorisation 2024": "freq_voitures"})

#convert string to float
colonnes = ["freq_voitures"]

taux_motorisation_communes[colonnes] = taux_motorisation_communes[colonnes].apply(pd.to_numeric, errors='coerce')

In [ ]:
taux_motorisation_communes.head()

In [ ]:
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.merge(
    taux_motorisation_communes,
    left_on="NO_COM_FED",
    right_on="Code",
    how="left" 
)

In [ ]:
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.drop(columns=['Code', 'Libellé'])

In [ ]:
zones_communes_GE_fusionnee.head()

**[CHOMAGE](https://statistique.ge.ch/atlas/index.php#c=indicator&view=map3)**

In [ ]:
chomage_communes = pd.read_csv(f'{input_file_path}/STAT/CHOMAGE/chomage_2025.csv', sep=";", header=2)

In [ ]:
chomage_communes.head()

In [ ]:
chomage_communes = chomage_communes.rename(columns={"Chômeurs inscrits 2025": "nb_chomage"})

#convert string to float
colonnes = ["nb_chomage"]

chomage_communes[colonnes] = chomage_communes[colonnes].apply(pd.to_numeric, errors='coerce')

In [ ]:
chomage_communes.head()

In [ ]:
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.merge(
    chomage_communes,
    left_on="NO_COM_FED",
    right_on="Code",
    how="left" 
)

In [ ]:
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.drop(columns=['Code', 'Libellé'])

In [ ]:
zones_communes_GE_fusionnee.head()

**EXPORT**

In [ ]:
zones_communes_GE_fusionnee.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_communes_GE_fusionnee_CRIM.gpkg"), driver="GPKG")
zones_communes_GE_fusionnee.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_communes_GE_fusionnee_CRIM.parquet')

**CORRELATION**

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

colonnes_corr = ['walk_index', 
                 'freq_LStup_infra', 
                 'freq_CP_infra', 
                 'freq_LEI_infra', 
                 'freq_crim_mean', 
                 'freq_voitures',
                 'nb_chomage']

zones_communes_GE_fusionnee_corr = zones_communes_GE_fusionnee[colonnes_corr].corr(method='pearson')
#print(zones_communes_corr)

sns.heatmap(zones_communes_GE_fusionnee_corr, annot=True, cmap='coolwarm')
plt.title("Pearson Correlation Heatmap")
plt.show()